# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.co

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'company homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'related project page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'},
  {'type': 'Blog page', 'url': 'https://edwarddonner.com/posts/'}]}

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [10]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 25 relevant links


{'links': [{'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'models catalog', 'url': 'https://huggingface.co/models'},
  {'type': 'datasets catalog', 'url': 'https://huggingface.co/datasets'},
  {'type': 'spaces catalog', 'url': 'https://huggingface.co/spaces'},
  {'type': 'docs page', 'url': 'https://huggingface.co/docs'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'endpoints API', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter', 'url': 

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [11]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-35B-A3B
Updated
3 days ago
•
587k
•
777
Qwen/Qwen3.5-27B
Updated
5 days ago
•
260k
•
492
unsloth/Qwen3.5-35B-A3B-GGUF
Updated
3 days ago
•
503k
•
422
Qwen/Qwen3.5-122B-A10B
Updated
about 7 hours ago
•
134k
•
367
Qwen/Qwen3.5-397B-A17B
Updated
7 days ago
•
1.14M
•
1.16k
Browse 2M+ models
Spaces
Running
on
Zero
Featured
1.77k
Qwen Image Multiple Angles 3D Camera
🎥
1.77k
Change the camera angle of a photo with AI
Running
on
Zero
Featured
266
Omni Video Factory
🏆
266
text to video, image to video, video extend
Running
on
Zero
MCP
1.03k
Wan2.2 14B Preview
🐌
1.03k
generate a video from an image with a text prompt
Running
o

In [20]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [21]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [22]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-35B-A3B\nUpdated\n3 days ago\n•\n587k\n•\n778\nQwen/Qwen3.5-27B\nUpdated\n5 days ago\n•\n260k\n•\n492\nunsloth/Qwen3.5-35B-A3B-GGUF\nUpdated\n3 days ago\n•\n503k\n•\n423\nQwen/Qwen3.5-122B-A10B\nUpdated\nabout 7 hours ago\n•\n134k\n•\n367\nQwen/Qwen3.5-397B-A17B\nUpdated\n7 days ago\n•\n1.14M\n•\n1.16k\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n1.77k\nQwen Image M

In [23]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [24]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 10 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Welcome to Hugging Face: The AI Community Building the Future 🚀🤗

---

## Who Are We?  
Hugging Face isn’t just a company—it’s a buzzing community where machine learning enthusiasts, scientists, and engineers unite to **collaborate, innovate, and share** open-source AI models, datasets, and applications. Think of us as a giant playground for AI geeks and creators—only with less mud and more code.  

We’re the *go-to platform* for exploring everything from text and images to audio, video—and even 3D models. Our mission? To empower you to build the future of AI ethically and openly, one model at a time.

---

## What’s in Our Magic AI Toolbox?  
- **2 MILLION+ Models:** Browse state-of-the-art machine learning models—from clever chatbots to jaw-dropping image generators.  
- **500K+ Datasets:** Dive into a treasure trove of datasets to train and test your next big AI creation.  
- **1 MILLION+ AI Applications:** Discover cutting-edge apps crafted by the community—because why reinvent the wheel?  
- **Spaces:** Run your AI apps directly on our platform and show off your AI wizardry without breaking a sweat.  

---

## The Hugging Face Vibes 😎  
- **Collaboration is King:** Share your models and datasets publicly, get feedback, and build your own ML portfolio all in one cozy hub.  
- **Open Source Awesomeness:** We’re powered by an open-source stack because great AI should be free and accessible (and hey, sharing is caring!).  
- **Ethics at the Core:** AI is cool, but ethical AI is cooler. We’re passionate about building an AI future everyone can trust.  
- **A Community Larger Than Your Grandma’s Cookie Jar:** Thousands of coders just like you are actively updating, tweaking, and improving AI models every day.  

---

## Why Join Us?  
### For Customers & Tech Explorers:  
- Gain access to the latest in AI technology without having to build everything from scratch.  
- Tap into vibrant forums and resources where the best AI minds share insights.  
- Scale your enterprise AI projects with Hugging Face’s robust tools and paid compute options.  

### For Job Seekers & AI Dreamers:  
- Join a forward-thinking team that values innovation, openness, and a dash of fun.  
- Work at the cutting edge of AI with generous opportunities to contribute to global open source projects.  
- Build your career where your work makes a tangible impact on the future of machine learning.  

---

## What the Cool Kids Are Using Right Now  
- **Qwen3.5-35B-A3B**: The big brain powering over half a million interactions recently.  
- **Qwen Image Multiple Angles 3D Camera:** Change the camera angle of photos with neat AI tricks—because who doesn’t want to be a photography wizard?  
- **Omni Video Factory:** Transform text and images into videos—movie magic with machine learning!  
- **Qwen3-TTS Demo:** Generate smooth custom speech from text or voice samples; Alexa, who?  

---

## Join the Future Today!  
Visit [Hugging Face](https://huggingface.co) and become part of the most welcoming, ambitious, and playful AI community on the planet. Whether you’re a code ninja, a data diva, or an AI dreamer, there’s a cozy spot here just for you.

**Hugging Face**  
_Where machines learn—and so do we!_ 🤗

---

*P.S. We promise our AI models won’t steal your job... yet. But maybe join us before they do!*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [25]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [26]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


# Welcome to Hugging Face: The AI Playground Where Future is Made Today! 🤗🚀

---

## Who We Are

At **Hugging Face**, we’re not just a company — we’re the **global AI community** where machine learning magicians, data wizards, and model maestros collaborate to build the future. Fancy playing with over **2 million AI models, 500k+ datasets, and 1 million+ AI applications**? You’re in the right place!

We’re the playground for the brightest minds in AI, making machine learning accessible, collaborative, and — dare we say — fun.

---

## What We Do

- **Explore & Share AI Models:** From text to images, videos, audio and even 3D — we host a staggering collection of cutting-edge models. Imagine changing camera angles on your photos or generating high-quality images from text prompts. Want speech generated from your own voice description? We got you.
  
- **Datasets Galore:** Unlock access to hundreds of thousands of datasets updated regularly by the community. From reasoning filters to coding data, we have something for every AI appetite.

- **Spaces:** Launch, host, and share your ML demos and interactive applications easily. Think of it as your AI app store meet collaboration hub.

---

## Culture & Community

Our DNA? **Open, collaborative, and wildly innovative.** We’re a vibrant bunch, driven by the excitement of open-source spirit with a dash of AI geekiness. Whether you’re a coding ninja, a dataset collector, or someone eager to learn the ropes, Hugging Face welcomes you with a high-five and a Hugging Face emoji 🤗.

- **Collaborate Without Limits:** Unlimited public models, datasets & apps to host & share.
- **Build Your ML Portfolio:** Share projects, gather feedback, and grow stellar reputations.
- **Community-Driven:** Engage with thousands of creators and AI enthusiasts worldwide.

---

## Business & Enterprise? We Got That Too! 💼

For teams and big brains building big things, Hugging Face offers:

- **Dedicated Enterprise Hub:** Enterprise-grade security, Single Sign-On (SSO), granular access permissions, audit logs, private datasets view – basically the AI fortress your company deserves.
  
- **Flexible Pricing:** Teams start at $20/user/month, with premium compute resources like ZeroGPU Quota Boost to power your projects faster than you can say “machine learning.”

- **Priority Support & Analytics:** Keep your AI factory humming with real-time analytics and expert help on speed dial.

---

## Pricing Highlights (Because We Love Transparency!)

- **PRO Account for Individuals:** $9/month to level up your personal AI playground with private storage, inference credits, priority hosting, and a shiny Pro badge to brag with.
  
- **Team Plans:** From $20 per user/month, instantly hook your team up with premium tools and collaboration options.

Enterprise customers? Let's chat — flexible contracts and custom solutions await to boost your AI game to cosmic levels.

---

## Careers – Join the Hugging Revolution! 🌟

If you love AI, open-source, and community vibes more than coffee (or at least as much), Hugging Face just might be your dream job destination.

You’ll work alongside world-class AI pros, contribute to open source that’s changing the world, and get to play with the coolest tech before it’s cool.

Open roles span Engineering, Research, Product, Community, and more — because hugging is what we do, but innovation is how we hug the future.

---

## Why Hugging Face?

- **Future-Facing**: Join the AI community literally *building the future.*
- **Open & Collaborative**: Share your creations. Discover others. Make AI magic together.
- **Inclusive**: For hobbyists, students, startups, and global enterprises alike.
- **Cutting Edge**: Stay ahead with the latest models, datasets, and compute resources.

---

## Ready to Hug the Future?

Dive in at [huggingface.co](https://huggingface.co) — where AI is not just smart, it’s friendly too!

Join us, build something incredible, and remember: here at Hugging Face, every model deserves a hug. 🤗

---

*Hugging Face: AI Collaboration, Elevated.*

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>